In [1]:
import os
import ssl
import requests
import urllib3
from transformers import pipeline

# 1. Handle the SSL/Firewall block
ssl._create_default_https_context = ssl._create_unverified_context
os.environ['CURL_CA_BUNDLE'] = ''
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# 2. Simplified Pipeline Call
# We removed model_kwargs to avoid the "multiple values" TypeError
model_name = "distilbert-base-uncased-finetuned-sst-2-english"

print("Downloading model...")
classifier = pipeline(
    "sentiment-analysis", 
    model=model_name
)

# 3. Test
print(classifier("The app is not bad"))

c:\Users\Almazt\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


RuntimeError: At least one of TensorFlow 2.0 or PyTorch should be installed. To install TensorFlow 2.0, read the instructions at https://www.tensorflow.org/install/ To install PyTorch, read the instructions at https://pytorch.org/.

In [ ]:
import os
import ssl
import requests
from transformers import pipeline

# 1. Force Python to ignore the self-signed certificate error
ssl._create_default_https_context = ssl._create_unverified_context
os.environ['CURL_CA_BUNDLE'] = ''

# 2. Patch the 'requests' library (which Hugging Face uses) to skip verification
# This is a bit "hacky" but works 99% of the time in corporate environments
from functools import partial
requests.Session.request = partial(requests.Session.request, verify=False)

# 3. Now try to initialize your DistilBERT pipeline
model_name = "distilbert-base-uncased-finetuned-sst-2-english"

try:
    print("Attempting to download/load model...")
    classifier = pipeline("sentiment-analysis", model=model_name)
    
    # Test it
    result = classifier("The app is not bad")
    print(f"Success! Result: {result}")
    
except Exception as e:
    print(f"Still failing. Error: {e}")
    
from transformers import pipeline, AutoTokenizer

# 1. Initialize the pipeline and tokenizer
model_name = "distilbert-base-uncased-finetuned-sst-2-english"
classifier = pipeline("sentiment-analysis", model=model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

def smart_truncate(text):
    # Tokenize and truncate to 512 tokens, then convert back to string
    tokens = tokenizer(text, truncation=True, max_length=512, add_special_tokens=True)
    return tokenizer.decode(tokens['input_ids'], skip_special_tokens=True)

def analyze_review(review):
    # 2. Handle proper token-based truncation
    truncated_review = smart_truncate(review)
    
    # 3. Perform inference
    result = classifier(truncated_review)
    return result[0]

# Examples
reviews = [
    "The app is not bad", 
    "The app is bad",
    "This is the most incredible experience I have ever had using a mobile interface!"
]

# Run the analyzer
for r in reviews:
    analysis = analyze_review(r)
    print(f"Review: {r}")
    print(f"Label: {analysis['label']}, Score: {analysis['score']:.4f}\n")
# Pro-Tip on Hugging Face Pipelines:
# If you are passing your data to the Hugging Face pipeline anyway, it can actually handle truncation internally for you! If you want to make your code even shorter, you can pass parameters directly into the classifier object like this:

# Alternative super-clean inference step
result = classifier(reviews, truncation=True, max_length=512)